# Missing `# needs sage.X` Guards — Why CI Fails on the Wrong Line

**What this notebook teaches:**
- What `# needs sage.X` does in passagemath doctests
- Why a missing guard causes a `NameError` on a completely unrelated line
- How to add guards correctly (inline and block forms)
- How to find the right tag name

**Scope:** this is narrower than the ImportError exercise. The ImportError pattern teaches something about Python itself — name binding, scoping — that transfers to any project. This teaches a doctest annotation convention specific to passagemath. It's worth knowing if you're writing or editing doctests, but don't expect a generalizable insight underneath it.

**The pattern:** same root cause as notebook 1 — a dep is absent, something silently doesn't happen, confusing error later. This is the doctest version.

**Prerequisites:** familiarity with passagemath doctests (`sage:` prompt). If you haven't worked through the ImportError pattern yet, [start there first](../importerror-fix-pattern/importerror-fix-pattern.ipynb).

**Time:** ~25 minutes.

In [ ]:
# Setup — run this first.
# Requires the 'passagemath (needs-sage-guards)' kernel.
import sys
import re
assert sys.version_info >= (3, 11), (
    f"Wrong kernel — expected Python 3.11+, got {sys.version_info}"
)

def simulate_doctest(src, available=frozenset()):
    """
    Simulate how passagemath's doctest runner processes # needs guards.

    Each 'sage:' line is labelled:
      RUN   - will execute
      SKIP  - guarded by '# needs X' where X is not in `available`
      BLOCK - a block guard 'sage: # needs X' that skips all following lines

    This parses line patterns only — it does not execute any code.
    Limitation: only the first tag on a # needs line is processed.
    Lines like '# needs sage.groups sage.modules' are treated as '# needs sage.groups'.
    """
    block_needs = None
    for raw in src.strip().split('\n'):
        line = raw.strip()
        if not line.startswith('sage:'):
            continue
        # Block guard: sage: # needs X
        bm = re.match(r'sage:\s*#\s*needs\s+(\S+)\s*$', line)
        if bm:
            block_needs = bm.group(1)
            tag = 'BLOCK' if block_needs not in available else 'BLOCK(run)'
            print(f"  {tag:<12}  {line}")
            continue
        # Inline guard (or inherited block guard)
        im = re.search(r'#\s*needs\s+(\S+)', line)
        needed = im.group(1) if im else block_needs
        tag = 'SKIP' if (needed and needed not in available) else 'RUN'
        print(f"  {tag:<12}  {line}")

print("Setup OK.")

---
## Section 1: The Crime Scene

You're working on `automatic_semigroup.py`. Doctests pass locally. You push. CI sends back:

```
NameError: name 'S5' is not defined
```

You open the file. The failing line is `S5.rename()`. Just above it:

```
sage: S5 = SymmetricGroup(5); S5.rename('S5')          # needs sage.groups
sage: AutomaticSemigroup(Family({1: S5((1,2))}),       # needs sage.groups
....:                    category=Groups().Finite().Subobjects())
A subgroup of (S5) with 1 generators
sage: S5.rename()
```

`S5` is assigned right there on line one. How is it not defined?

*(Issue [#2256](https://github.com/passagemath/passagemath/issues/2256).)*

---
## Section 2: What `# needs` Does

**The rule:** if `sage.groups` is not installed in the test environment, any line tagged `# needs sage.groups` is silently skipped — not run, not errored, just absent.

Run the cell below.

In [ ]:
# A simple doctest with one # needs tag.
# See what happens when sage.groups is absent vs. present.

_demo = """
    sage: x = 1 + 1
    sage: x
    2
    sage: S5 = SymmetricGroup(5)    # needs sage.groups
    sage: S5
    Symmetric group of order 5! as a permutation group
"""

print("=== sage.groups ABSENT ===")
simulate_doctest(_demo, available=set())

print()
print("=== sage.groups PRESENT ===")
simulate_doctest(_demo, available={'sage.groups'})

When `sage.groups` is absent: the `S5 = SymmetricGroup(5)` line is skipped. No error, no message. The test runner moves on.

The problem happens when the skipped line was supposed to bind a name that a later, **unguarded** line uses.

---
## Section 3: The Trap

Run the cell below. This is the actual pre-fix code from `automatic_semigroup.py`.

In [ ]:
# Pre-fix state — issue #2256.
# Setup lines are guarded. The teardown line is not.

_pre_fix = """
    sage: S5 = SymmetricGroup(5); S5.rename('S5')          # needs sage.groups
    sage: AutomaticSemigroup(Family({1: S5((1,2))}),       # needs sage.groups
    sage: S5.rename()
"""

print("=== sage.groups ABSENT (CI) ===")
simulate_doctest(_pre_fix, available=set())

print()
# The setup lines were skipped — S5 was never bound.
# This is the actual error when the unguarded line runs:
try:
    _S5_not_defined.rename()
except NameError as e:
    print(type(e).__name__ + ":", e)

Lines 1–2: `SKIP`. `S5` is never assigned.

Line 3: `RUN`. `S5.rename()` executes. `S5` is not defined. `NameError: name 'S5' is not defined`.

The error lands on the cleanup call, not the setup. Nothing in the message mentions `sage.groups`. If you only look at the failing line, it looks like a completely unrelated bug.

The actual problem: the teardown line is missing its guard.

---
## Section 4: The Fix

Before reading on — what should you add to the teardown line?

```
sage: S5.rename()    # ← what goes here?
```

Edit the cell below to add the guard. You've got it when `sage.groups` absent shows all three lines as SKIP.

In [ ]:
# YOUR FIX: add the missing guard to the teardown line.
# Goal: sage.groups absent → all three lines SKIP.

_your_fix = """
    sage: S5 = SymmetricGroup(5); S5.rename('S5')          # needs sage.groups
    sage: AutomaticSemigroup(Family({1: S5((1,2))}),       # needs sage.groups
    sage: S5.rename()
"""

print("=== sage.groups ABSENT ===")
simulate_doctest(_your_fix, available=set())
print()
print("=== sage.groups PRESENT ===")
simulate_doctest(_your_fix, available={'sage.groups'})

The fix: `# needs sage.groups` on the teardown line.

```
sage: S5.rename()   # needs sage.groups
```

This is the **inline guard** form — one tag, one line. When `sage.groups` is absent, skip this line. When present, run it.

In [ ]:
# Post-fix. All three lines guarded.
# Absent → all skip. Present → all run.

_fixed = """
    sage: S5 = SymmetricGroup(5); S5.rename('S5')          # needs sage.groups
    sage: AutomaticSemigroup(Family({1: S5((1,2))}),       # needs sage.groups
    sage: S5.rename()                                       # needs sage.groups
"""

print("=== sage.groups ABSENT ===")
simulate_doctest(_fixed, available=set())
print()
print("=== sage.groups PRESENT ===")
simulate_doctest(_fixed, available={'sage.groups'})

---
## Section 5: Block Form

When an entire doctest block depends on one dep, inline guards on every line get noisy. There's a cleaner form:

```
sage: # needs sage.groups
sage: S5 = SymmetricGroup(5); S5.rename('S5')
sage: AutomaticSemigroup(...)
sage: S5.rename()
```

The `sage: # needs sage.groups` line is a **block guard** — it applies to all remaining `sage:` lines in the same docstring. A different method's docstring starts fresh.

Use **inline** when only specific lines in a mixed test need the guard. Use **block** when the whole test depends on the dep.

In [ ]:
# Block form — one guard covers all following lines.

_block = """
    sage: # needs sage.groups
    sage: S5 = SymmetricGroup(5); S5.rename('S5')
    sage: AutomaticSemigroup(Family({1: S5((1,2))}),
    sage: S5.rename()
"""

print("=== sage.groups ABSENT ===")
simulate_doctest(_block, available=set())
print()
print("=== sage.groups PRESENT ===")
simulate_doctest(_block, available={'sage.groups'})

---
## Section 6: Finding the Right Tag

Tag names mirror module paths: `sage.libs.pari`, `sage.groups`, `sage.modules`. Every optional dep has a Feature class in `sage.features`. The `.name` attribute gives the exact tag to write.

In [ ]:
from sage.features.sagemath import sage__libs__pari, sage__libs__flint, sage__modules, sage__groups

for f in [sage__libs__pari(), sage__libs__flint(), sage__modules(), sage__groups()]:
    present = bool(f.is_present())
    print(f"  {f.__class__.__name__:<28}  .name = {f.name!r:<22}  present: {present}")

The class uses double underscores (`sage__libs__pari`). The `.name` uses dots (`sage.libs.pari`). The `.name` is what goes in the `# needs` comment.

Note that `sage.modules` is present here — `passagemath-modules` is installed in this venv. `sage.groups` is absent. This environment models CI without groups installed.

---
## Section 7: Exercise — `polynomial_quotient_ring.py`

PR [#2293](https://github.com/passagemath/passagemath/pull/2293) fixed three doctests in `_element_constructor_` that were missing `# needs sage.modules`.

The string-conversion path in `_element_constructor_` requires `sage.modules`. PR #2069 added `# needs sage.modules` guards to most affected call sites but missed these three. PR #2293 caught them when the test-mod CI job started failing.

Here are the three pre-fix doctests. They are fragments of the same docstring in the real source — `P`, `Q1`, and `Q2` are defined earlier in the same block. `simulate_doctest` parses patterns only, so the missing context doesn't affect the exercise.

Your task: add `# needs sage.modules` guards so that when `sage.modules` is absent, the string-conversion lines show SKIP instead of RUN.

Hint: doctest 2 has four string-conversion lines in a row — all need the same guard, and there's a cleaner option than tagging each one individually.

Try to fix each one before reading the answer.

In [ ]:
# Doctest 1 of 3 — inline case.
# p.parent()('xbar') calls string-to-element conversion via sage.modules.
# Add a guard so it shows SKIP when sage.modules is absent.

_ex1 = """
    sage: P.<x> = QQ[]
    sage: Q1 = P.quo([(x**2+1)**2*(x**2-3)])
    sage: Q2 = P.quo([(x**2+1)**2*(x**5+3)])
    sage: p = Q1.gen() + Q2.gen()
    sage: p.parent()('xbar')
    xbar
"""

# Goal: p.parent()('xbar') line shows SKIP
print("=== sage.modules ABSENT ===")
simulate_doctest(_ex1, available=set())

In [ ]:
# Doctest 2 of 3 — block case.
# Q1 is defined earlier in the same docstring (see _ex1 above).
# All four sage: lines call string-to-element conversion via sage.modules.
# Add one guard that covers all four lines.

_ex2 = """
    sage: a = Q1('x'); a
    xbar
    sage: a.parent() is Q1
    True
    sage: b = Q1('1'); b
    1
    sage: b.parent() is Q1
    True
"""

# Goal: all four sage: lines show SKIP
print("=== sage.modules ABSENT ===")
simulate_doctest(_ex2, available=set())

In [ ]:
# Doctest 3 of 3 — inline case.
# P and Q1 are defined earlier in the same docstring (see _ex1 above).
# Q3('x*ybar^2') calls string-to-element conversion via sage.modules.
# (^ is Sage's power syntax — the preparser converts it to **.)

_ex3 = """
    sage: R.<y> = P[]
    sage: Q3 = R.quo([(y**2+1)])
    sage: Q3(Q1.gen())
    x
    sage: Q3('x*ybar^2')
    -x
"""

# Goal: Q3('x*ybar^2') line shows SKIP
print("=== sage.modules ABSENT ===")
simulate_doctest(_ex3, available=set())

---
### Answer — from the actual PR #2293 diff

**Doctest 1:** inline guard on the string-conversion line:

```diff
-            sage: p.parent()('xbar')
+            sage: p.parent()('xbar')                                          # needs sage.modules
             xbar
```

**Doctest 2:** block guard — the whole block is one test, cleaner to guard once:

```diff
+            sage: # needs sage.modules
             sage: a = Q1('x'); a
             xbar
             ...
```

**Doctest 3:** inline guard:

```diff
-            sage: Q3('x*ybar^2')
+            sage: Q3('x*ybar^2')                                              # needs sage.modules
             -x
```

Once each doctest shows the string-conversion lines as SKIP when `sage.modules` is absent, you've matched the [actual PR diff](https://github.com/passagemath/passagemath/pull/2293).

---
## Summary

```
sage: line_of_code()   # needs sage.X        ← inline: this line only
sage: # needs sage.X                         ← block: all following lines
```

**The trap:** guard the setup, miss the teardown → teardown runs on an undefined name → error on the wrong line, no mention of the missing dep.

**Finding the tag:** `from sage.features.sagemath import sage__X; sage__X().name`

**Inline vs. block:** block when the whole test depends on one dep (cleaner, harder to miss).

**Connection to notebook 1:** same category of failure — dep absent, something silently doesn't happen, confusing error later. Notebook 1 is a runtime crash (`NameError` when calling a function). This is a CI test failure (`NameError` in a doctest). The root cause logic is identical.